In [ ]:
# [Célula 01] Setup + carregar datamart (wide aluno-ano)
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# tenta localizar o arquivo em locais comuns do repo
CANDIDATES = [
    "datamart_wide_aluno_ano.csv",
    "../data/datamart_wide_aluno_ano.csv",
    "../datamart_wide_aluno_ano.csv",
    "./data/datamart_wide_aluno_ano.csv",
]

DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_PATH is not None, f"Não encontrei datamart_wide_aluno_ano.csv. Tentei: {CANDIDATES}"

panel = pd.read_csv(DATA_PATH)
print("✅ Carregado:", DATA_PATH, "| shape:", panel.shape)
display(panel.head(3))


✅ Carregado: datamart_wide_aluno_ano.csv | shape: (3030, 18)


,RA,NOME,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,ANO_INGRESSO,IAN,IDADE,IEG,IAA,IPS,IPV,INDE,PEDRA,ANO,IPP,DEFASAGEM
0,RA-1,Aluno-1,7,A,Menina,Escola Pública,2016,5.0,19,4.1,8.3,5.6,7.278,5.783,Quartzo,2022,NaN,NaN
1,RA-2,Aluno-2,7,A,Menina,Rede Decisão,2017,10.0,17,5.2,8.8,6.3,6.778,7.055,Ametista,2022,NaN,NaN
2,RA-3,Aluno-3,7,A,Menina,Rede Decisão,2016,10.0,17,7.9,0.0,5.6,7.556,6.591,Ágata,2022,NaN,NaN


In [ ]:
# [Célula 02] Checagens + tipos básicos (RA/ANO) + colunas essenciais
required = ["RA", "ANO", "IAN"]
missing = [c for c in required if c not in panel.columns]
assert not missing, f"Faltando colunas essenciais: {missing}"

panel["RA"] = panel["RA"].astype(str).str.strip()
panel["ANO"] = pd.to_numeric(panel["ANO"], errors="coerce").astype("Int64")

# remove linhas sem RA/ANO
panel = panel.dropna(subset=["RA", "ANO"]).copy()
panel["ANO"] = panel["ANO"].astype(int)

print("✅ Tipos OK | ANO min/max:", panel["ANO"].min(), panel["ANO"].max())
print("✅ RAs únicos:", panel["RA"].nunique())
display(panel[["RA","ANO","IAN"]].head(5))


✅ Tipos OK | ANO min/max: 2022 2024
✅ RAs únicos: 1661


,RA,ANO,IAN
0,RA-1,2022,5.0
1,RA-2,2022,10.0
2,RA-3,2022,10.0
3,RA-4,2022,10.0
4,RA-5,2022,10.0


In [ ]:
# [Célula 03] Padronizar tipos: numéricas -> float, categóricas -> string
# (ajuste a lista se você quiser incluir/excluir features do datamart)

CAT_BASE = ["FASE", "TURMA", "GENERO", "INSTITUICAO_DE_ENSINO", "PEDRA"]
NUM_BASE = ["ANO_INGRESSO", "IDADE", "INDE", "IEG", "IAA", "IPS", "IPV", "IPP", "DEFASAGEM", "IAN"]

# mantém só as que existem de fato no seu panel
CAT_BASE = [c for c in CAT_BASE if c in panel.columns]
NUM_BASE = [c for c in NUM_BASE if c in panel.columns]

# categóricas
for c in CAT_BASE:
    panel[c] = panel[c].astype(str).str.strip()

# numéricas
for c in NUM_BASE:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

print("Categóricas:", CAT_BASE)
print("Numéricas:", NUM_BASE)
display(panel[["RA","ANO"] + CAT_BASE + NUM_BASE].head(3))


Categóricas: ['FASE', 'TURMA', 'GENERO', 'INSTITUICAO_DE_ENSINO', 'PEDRA']
Numéricas: ['ANO_INGRESSO', 'IDADE', 'INDE', 'IEG', 'IAA', 'IPS', 'IPV', 'IPP', 'DEFASAGEM', 'IAN']


,RA,ANO,FASE,TURMA,GENERO,INSTITUICAO_DE_ENSINO,PEDRA,ANO_INGRESSO,IDADE,INDE,IEG,IAA,IPS,IPV,IPP,DEFASAGEM,IAN
0,RA-1,2022,7,A,Menina,Escola Pública,Quartzo,2016,19.0,5.783,4.1,8.3,5.6,7.278,NaN,NaN,5.0
1,RA-2,2022,7,A,Menina,Rede Decisão,Ametista,2017,17.0,7.055,5.2,8.8,6.3,6.778,NaN,NaN,10.0
2,RA-3,2022,7,A,Menina,Rede Decisão,Ágata,2016,17.0,6.591,7.9,0.0,5.6,7.556,NaN,NaN,10.0


In [ ]:
# [Célula 04] Criar TARGET correto (prever ano t usando dados do ano t-1) + features defasadas (LAG1)
df = panel.copy()
df = df.sort_values(["RA", "ANO"]).reset_index(drop=True)

# target: IAN(t) > IAN(t-1)
df["IAN_PREV"] = df.groupby("RA")["IAN"].shift(1)
df["TARGET_RISCO_IAN"] = (df["IAN"] > df["IAN_PREV"]).astype("Int64")

# cria features do ano anterior (lag1) para TODAS as features base (cat + num)
LAG_FEATURES = CAT_BASE + [c for c in NUM_BASE if c != "IAN"] + ["IAN"]  # IAN_LAG1 é permitido (histórico)
for c in LAG_FEATURES:
    df[f"{c}_LAG1"] = df.groupby("RA")[c].shift(1)

# mantém apenas linhas onde existe histórico (ou seja, tem lag)
df_ml = df.dropna(subset=["IAN_PREV"]).copy()

print("✅ df_ml shape:", df_ml.shape)
print("Distribuição do target:")
display(df_ml["TARGET_RISCO_IAN"].value_counts(normalize=True).rename("proportion"))
display(df_ml[["RA","ANO","IAN_PREV","IAN","TARGET_RISCO_IAN"]].head(8))


✅ df_ml shape: (1369, 35)
Distribuição do target:


,proportion
TARGET_RISCO_IAN,
0,0.790358
1,0.209642


,RA,ANO,IAN_PREV,IAN,TARGET_RISCO_IAN
1,RA-1,2023,5.0,10.0,1
2,RA-1,2024,10.0,10.0,0
6,RA-1000,2024,10.0,10.0,0
8,RA-1001,2024,5.0,5.0,0
10,RA-1002,2024,5.0,5.0,0
12,RA-1003,2024,10.0,5.0,0
14,RA-1004,2024,10.0,5.0,0
17,RA-1006,2024,10.0,5.0,0


In [ ]:
# [Célula 05] Montar X/y (somente LAG1 + ANO atual) e remover colunas de vazamento
# Observação: usar ANO (t) como feature é OK (você sabe o ano em que quer prever)

y = df_ml["TARGET_RISCO_IAN"].astype(int)

# features = todas as colunas *_LAG1 + ANO
lag_cols = [c for c in df_ml.columns if c.endswith("_LAG1")]
X = df_ml[["ANO"] + lag_cols].copy()

print("✅ X shape:", X.shape, "| y shape:", y.shape)
display(X.head(3))


✅ X shape: (1369, 16) | y shape: (1369,)


,ANO,FASE_LAG1,TURMA_LAG1,GENERO_LAG1,INSTITUICAO_DE_ENSINO_LAG1,PEDRA_LAG1,ANO_INGRESSO_LAG1,IDADE_LAG1,INDE_LAG1,IEG_LAG1,IAA_LAG1,IPS_LAG1,IPV_LAG1,IPP_LAG1,DEFASAGEM_LAG1,IAN_LAG1
1,2023,7,A,Menina,Escola Pública,Quartzo,2016.0,19.0,5.7830,4.1,8.3,5.60,7.278,NaN,NaN,5.0
2,2024,FASE 8,8E,Feminino,Privada *Parcerias com Bolsa 100%,nan,2016.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,10.0
6,2024,ALFA,ALFA U - G2/G3,Feminino,Pública,Ametista,2023.0,8.0,7.9162,9.4,8.5,3.77,8.920,6.25,0.0,10.0


In [ ]:
# [Célula 06] Identificar colunas categóricas vs numéricas (agora no dataset final)
cat_cols = [c for c in X.columns if c.endswith("_LAG1") and any(base in c for base in CAT_BASE)]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categóricas (X):", cat_cols)
print("Numéricas (X):", num_cols)
print("\nTipos em X:")
display(X.dtypes.value_counts())


Categóricas (X): ['FASE_LAG1', 'TURMA_LAG1', 'GENERO_LAG1', 'INSTITUICAO_DE_ENSINO_LAG1', 'PEDRA_LAG1']
Numéricas (X): ['ANO', 'ANO_INGRESSO_LAG1', 'IDADE_LAG1', 'INDE_LAG1', 'IEG_LAG1', 'IAA_LAG1', 'IPS_LAG1', 'IPV_LAG1', 'IPP_LAG1', 'DEFASAGEM_LAG1', 'IAN_LAG1']

Tipos em X:


,count
float64,10
object,5
int64,1


In [ ]:
# [Célula 07] Split treino/teste (estratificado) + baseline da distribuição
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("✅ Train:", X_train.shape, "| Test:", X_test.shape)
print("Target train %:", y_train.value_counts(normalize=True).to_dict())
print("Target test  %:", y_test.value_counts(normalize=True).to_dict())


✅ Train: (1026, 16) | Test: (343, 16)
Target train %: {0: 0.7904483430799221, 1: 0.20955165692007796}
Target test  %: {0: 0.7900874635568513, 1: 0.2099125364431487}


In [ ]:
# [Célula 08] Preprocessamento + função de avaliação + pipelines (imports completos)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# preprocess
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  # só nas numéricas
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

def eval_model(pipe, X_te, y_te, name, threshold=0.5):
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= threshold).astype(int)

    out = {
        "model": name,
        "threshold": threshold,
        "auc": roc_auc_score(y_te, proba),
        "acc": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "f1": f1_score(y_te, pred, zero_division=0),
    }

    print(f"\n=== {name} | thr={threshold:.2f} ===")
    print("AUC:", round(out["auc"], 4),
          "| ACC:", round(out["acc"], 4),
          "| P:", round(out["precision"], 4),
          "| R:", round(out["recall"], 4),
          "| F1:", round(out["f1"], 4))

    print("\nConfusion matrix:\n", confusion_matrix(y_te, pred))
    print("\nReport:\n", classification_report(y_te, pred, zero_division=0))
    return out


In [ ]:
# [Célula 09] Treinar modelos + escolher melhor (AUC) e guardar best_pipe
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "logreg": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=None
    ),
    "rf": RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced_subsample"
    ),
}

trained = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)

    metrics = eval_model(pipe, X_test, y_test, name, threshold=0.5)
    trained[name] = {"pipe": pipe, **metrics}

# escolhe melhor por AUC (você pode trocar por f1 se quiser)
best_name = max(trained.keys(), key=lambda k: trained[k]["auc"])
best_pipe = trained[best_name]["pipe"]

print("\n✅ Melhor modelo (AUC):", best_name, "| AUC:", round(trained[best_name]["auc"], 4))



=== logreg | thr=0.50 ===
AUC: 0.9458 | ACC: 0.8601 | P: 0.6111 | R: 0.9167 | F1: 0.7333

Confusion matrix:
 [[229  42]
 [  6  66]]

Report:
               precision    recall  f1-score   support

           0       0.97      0.85      0.91       271
           1       0.61      0.92      0.73        72

    accuracy                           0.86       343
   macro avg       0.79      0.88      0.82       343
weighted avg       0.90      0.86      0.87       343


=== rf | thr=0.50 ===
AUC: 0.9319 | ACC: 0.8601 | P: 0.8333 | R: 0.4167 | F1: 0.5556

Confusion matrix:
 [[265   6]
 [ 42  30]]

Report:
               precision    recall  f1-score   support

           0       0.86      0.98      0.92       271
           1       0.83      0.42      0.56        72

    accuracy                           0.86       343
   macro avg       0.85      0.70      0.74       343
weighted avg       0.86      0.86      0.84       343


✅ Melhor modelo (AUC): logreg | AUC: 0.9458


In [ ]:
# [Célula 10] Ajuste de threshold (otimizar F1 e/ou Recall da classe 1)
import numpy as np
import pandas as pd

proba = best_pipe.predict_proba(X_test)[:, 1]

thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
rows = []
for thr in thresholds:
    pred = (proba >= thr).astype(int)
    rows.append({
        "thr": float(thr),
        "precision_1": precision_score(y_test, pred, zero_division=0),
        "recall_1": recall_score(y_test, pred, zero_division=0),
        "f1_1": f1_score(y_test, pred, zero_division=0),
        "pred_pos_rate": float(pred.mean()),
    })

tbl = pd.DataFrame(rows).sort_values("f1_1", ascending=False)
display(tbl.head(10))

best_f1 = tbl.iloc[0].to_dict()
best_rec = tbl.sort_values("recall_1", ascending=False).iloc[0].to_dict()

print("Melhor por F1:", best_f1)
print("Melhor por Recall:", best_rec)


,thr,precision_1,recall_1,f1_1,pred_pos_rate
10,0.60,0.673913,0.861111,0.756098,0.268222
11,0.65,0.714286,0.763889,0.738255,0.224490
7,0.45,0.600000,0.958333,0.737968,0.335277
9,0.55,0.636364,0.875000,0.736842,0.288630
8,0.50,0.611111,0.916667,0.733333,0.314869
12,0.70,0.726027,0.736111,0.731034,0.212828
13,0.75,0.738462,0.666667,0.700730,0.189504
14,0.80,0.875000,0.583333,0.700000,0.139942
6,0.40,0.534884,0.958333,0.686567,0.376093
5,0.35,0.514706,0.972222,0.673077,0.396501


Melhor por F1: {'thr': 0.6, 'precision_1': 0.6739130434782609, 'recall_1': 0.8611111111111112, 'f1_1': 0.7560975609756098, 'pred_pos_rate': 0.26822157434402333}
Melhor por Recall: {'thr': 0.1, 'precision_1': 0.3769633507853403, 'recall_1': 1.0, 'f1_1': 0.5475285171102662, 'pred_pos_rate': 0.5568513119533528}


In [ ]:
# [Célula 11] Salvar bundle (modelo + threshold) de forma reprodutível (sem depender de cache)
import os
import joblib

#  Defina o threshold final escolhido (fixo e explícito)
FINAL_THRESHOLD = 0.60

#  Monte o bundle padronizado
bundle = {
    "model": best_pipe,                 # pipeline completo (preprocess + modelo)
    "threshold": float(FINAL_THRESHOLD),
    "best_model_name": best_name,       # ex.: "logreg" / "rf"
    "target": "TARGET_RISCO_IAN",
    "positive_class": 1
}

# ✅ Salvar no padrão do repo (rodando dentro de notebooks/ ou fora)
os.makedirs("../models", exist_ok=True)
os.makedirs("models", exist_ok=True)

out_candidates = [
    "../models/ian_risk_pipeline.joblib",
    "models/ian_risk_pipeline.joblib",
]
OUT_PATH = out_candidates[0] if os.path.isdir("../models") else out_candidates[1]

joblib.dump(bundle, OUT_PATH)
print("✅ Bundle salvo em:", OUT_PATH)
print("✅ best_model:", bundle["best_model_name"], "| threshold:", bundle["threshold"])


✅ Bundle salvo em: ../models/ian_risk_pipeline.joblib
✅ best_model: logreg | threshold: 0.6


In [ ]:
# [Célula 12] Sanity check: carregar bundle e validar chaves
import joblib

bundle_loaded = joblib.load(OUT_PATH)

print("Tipo:", type(bundle_loaded))
print("Keys:", bundle_loaded.keys())

assert "model" in bundle_loaded, "Faltou 'model' no bundle"
assert "threshold" in bundle_loaded, "Faltou 'threshold' no bundle"

print("✅ OK - threshold:", bundle_loaded["threshold"])
print("✅ OK - model type:", type(bundle_loaded["model"]))


Tipo: <class 'dict'>
Keys: dict_keys(['model', 'threshold', 'best_model_name', 'target', 'positive_class'])
✅ OK - threshold: 0.6
✅ OK - model type: <class 'sklearn.pipeline.Pipeline'>
